# Fase 4: Incorporación de Demografía Real e Inmobiliaria
Sustituyendo modelos estocásticos (proxies) por mediciones empíricas procedentes del **Padrón Municipal (INE/Ayto)** y el **Registro Inmobiliario Real (€/m2)**.

In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar base principal
csv_path = '../data/processed/negocios_scored.csv'
df_base = pd.read_csv(csv_path)

def normalize_str(s):
    if pd.isna(s): return ''
    return str(s).upper().replace('Á', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U').strip()

df_base['barrio_norm'] = df_base['desc_barrio_local'].apply(normalize_str)
df_base['distrito_norm'] = df_base['desc_distrito_local'].apply(normalize_str)

## 1. Tratamiento del Padrón Municipal (Generaciones y Género)
Procesamos la columna edad para crear segmentaciones de mercado exactas:
- GenZ / Estudiantes: 15 a 24
- Profesionales / Millennials: 25 a 39
- Familias / Maduro: 40 a 64
- Seniors y Jubilados: 65+

In [ ]:
print("Procesando Padrón Municipal (Población real)...")
df_padron = pd.read_csv('../data/padron-csv.csv', sep=';')

# Normalizar nombres
df_padron['barrio_norm'] = df_padron['DESC_BARRIO'].apply(normalize_str)

# Limpiar edad
# La edad viene como cadena con espacios (ej. '        62'), pasamos a int
df_padron['edad_int'] = pd.to_numeric(df_padron['COD_EDAD_INT'], errors='coerce').fillna(0).astype(int)

# Crear grupos de edad
def get_grupo_edad(edad):
    if edad < 15: return 'child'
    elif edad <= 24: return 'genz'
    elif edad <= 39: return 'millennial'
    elif edad <= 64: return 'family'
    else: return 'senior'

df_padron['segmento_edad'] = df_padron['edad_int'].apply(get_grupo_edad)

# Calcular totales por fila
df_padron['total_hombres'] = df_padron['ESPANOLESHOMBRES'] + df_padron['EXTRANJEROSHOMBRES']
df_padron['total_mujeres'] = df_padron['ESPANOLESMUJERES'] + df_padron['EXTRANJEROSMUJERES']
df_padron['poblacion_total'] = df_padron['total_hombres'] + df_padron['total_mujeres']

# 1.1 Calcular porcentaje de mujeres por barrio
padron_genero = df_padron.groupby('barrio_norm')[['total_hombres', 'total_mujeres', 'poblacion_total']].sum().reset_index()
padron_genero['porcentaje_mujeres_real'] = (padron_genero['total_mujeres'] / padron_genero['poblacion_total']) * 100

# 1.2 Calcular distribución de edades por barrio (% de cada segmento)
padron_edades = df_padron.groupby(['barrio_norm', 'segmento_edad'])['poblacion_total'].sum().unstack(fill_value=0).reset_index()
padron_edades['pob_valida'] = padron_edades[['genz', 'millennial', 'family', 'senior']].sum(axis=1)

padron_edades['score_joven_real'] = (padron_edades['genz'] / padron_edades['pob_valida']) * 100
padron_edades['score_millennial_real'] = (padron_edades['millennial'] / padron_edades['pob_valida']) * 100
padron_edades['score_familiar_real'] = (padron_edades['family'] / padron_edades['pob_valida']) * 100
padron_edades['score_senior_real'] = (padron_edades['senior'] / padron_edades['pob_valida']) * 100

# Limpiar columnas para cruzar
padron_edades_lite = padron_edades[['barrio_norm', 'score_joven_real', 'score_millennial_real', 'score_familiar_real', 'score_senior_real']]
padron_genero_lite = padron_genero[['barrio_norm', 'porcentaje_mujeres_real', 'poblacion_total']]

# Join Padrón a base
df_base = pd.merge(df_base, padron_genero_lite, on='barrio_norm', how='left')
df_base = pd.merge(df_base, padron_edades_lite, on='barrio_norm', how='left')
print("Padrón integrado.")

## 2. Mercado Inmobiliario (€/m² exacto por Barrio)
Asimilamos la estadística oficial eliminando el heurístico de Renta.

In [ ]:
print("Procesando Precio M2...")
df_precios = pd.read_excel('../data/precios_m2_madrid.xlsx', skiprows=5)

# Renombrar columnas para claridad
cols = df_precios.columns.tolist()
df_precios.rename(columns={cols[1]: 'distrito_orig', cols[2]: 'barrio_orig', cols[3]: 'precio_m2_texto'}, inplace=True)

# Quedarnos solo con las filas que son de BARRIO (cuando barrio_orig != distrito_orig y no es Total ciudad)
df_precios = df_precios[df_precios['distrito_orig'] != df_precios['barrio_orig']].copy()

def extract_barrio_name(s):
    if pd.isna(s): return ''
    s = str(s)
    # Ejemplo: "011. Palacio" -> "Palacio"
    if '.' in s:
        return s.split('.', 1)[1].strip()
    return s.strip()

df_precios['barrio_limpio'] = df_precios['barrio_orig'].apply(extract_barrio_name)
df_precios['barrio_norm'] = df_precios['barrio_limpio'].apply(normalize_str)

# Convertir precio a float ('26,68' -> 26.68) -> cuidado con valores extraños o ".."
def parse_precio(val):
    try:
        if pd.isna(val) or val == '..': return np.nan
        val = str(val).replace('.', '').replace(',', '.') # Cuidado con miles si los hay, aquí típico es coma para decimales.
        return float(val)
    except:
        return np.nan

df_precios['precio_m2_real'] = df_precios['precio_m2_texto'].apply(parse_precio)
precios_lite = df_precios[['barrio_norm', 'precio_m2_real']].dropna()

# Join Precios a base
df_base = pd.merge(df_base, precios_lite, on='barrio_norm', how='left')

# Imputar valores nulos (media del distrito) si algún barrio falla
media_precio = df_base['precio_m2_real'].mean()
df_base['precio_m2_real'] = df_base['precio_m2_real'].fillna(media_precio)

print("Precios inmobiliarios integrados.")

## 3. Limpieza de Proxies Históricos y Guardado
Sustituimos las columnas `score_...` antiguas que eran generadas estocásticamente si existían.

In [ ]:
df_base.drop(columns=['barrio_norm', 'distrito_norm'], inplace=True, errors='ignore')

print('Variables reales nuevas:', [c for c in df_base.columns if '_real' in c])
print('Guardando base empírica V4 final...')

df_base.to_csv(csv_path, index=False)
print('¡ÉXITO! Predictor libre de proxies estocásticos demográficos e inmobiliarios.')